# ViT APJN experiments notebook

This notebook follows the `vit-dyt` bootstrap flow from the reference Colab before running the APJN experiments.

In [ ]:
# Optional: install TeX/font packages for publication-style plots.
# Run this cell only if you want the LaTeX/NeurIPS-like font setup below.
!apt-get -qq update
!apt-get -qq install texlive-latex-extra texlive-fonts-recommended dvipng cm-super

In [ ]:
# Optional: configure NeurIPS-like LaTeX fonts.
# Run after the apt-get cell above.
# If you want the simpler Times-like fallback instead, use `configure_times_like_tex_fonts()`.
from vit_apjn_notebook_helpers import configure_neurips_like_tex_fonts, configure_times_like_tex_fonts

configure_times_like_tex_fonts()

In [ ]:
# Colab/local setup for vit-dyt.
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path('vit-dyt')
INSTALL_TIMM = True
INSTALL_REQUIREMENTS = True
CLONE_IF_MISSING = True

if INSTALL_TIMM:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)

if not REPO_DIR.exists() and CLONE_IF_MISSING:
    subprocess.run(['git', 'clone', 'https://github.com/labofdoubt/vit-dyt', str(REPO_DIR)], check=True)

REQ = REPO_DIR / 'requirements.txt'
if INSTALL_REQUIREMENTS and REQ.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)], check=True)

if REPO_DIR.exists() and str(REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_DIR.resolve()))

print('Repo directory:', REPO_DIR.resolve() if REPO_DIR.exists() else 'missing')

In [ ]:
# Common imports and helper bootstrap.
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from vit_apjn_notebook_helpers import *

bootstrap_vit_dyt_repo(
    REPO_DIR,
    clone_if_missing=CLONE_IF_MISSING,
    install_requirements=INSTALL_REQUIREMENTS,
    install_timm=INSTALL_TIMM,
)
set_pub_style()
print('DEVICE =', DEVICE)

In [ ]:
# Global defaults and top-level tuning handles.
DEFAULT_DEPTH = 64
DEFAULT_ALPHAS = tuple(np.arange(0.1, 1.9 + 1e-9, 0.2).astype(float))
DEFAULT_Q0_GRID = tuple(np.linspace(0.0, 2.0, 41))
DEFAULT_P0_GRID = tuple(np.linspace(0.0, 2.0, 41))
DEFAULT_PRELN_SCALE_GRID = tuple(np.linspace(0.4, 8.0, 61))

MODEL_CFG = ModelConfig(
    model_name='vit_base_patch16_224',
    depth=DEFAULT_DEPTH,
    num_classes=100,
    img_size=224,
    replace_gelu_with_relu=True,
    inplace_relu=False,
    seed=0,
)
MEAN_FIELD_CFG = build_mean_field_cfg_for_vit_base(depth=MODEL_CFG.depth)
STYLE_CFG = FinalThreePanelStyleConfig(
    title_fs=18,
    label_fs=18,
    annotation_fs=18,
    colorbar_pad=0.04,
    )

# Direct plotting overrides for notebook figures.
PLOT_TICK_FS = 16
PLOT_LABEL_FS = 18
PLOT_TITLE_FS = 18
PLOT_ANNOTATION_FS = 18
PLOT_ALPHA_LEGEND_FS = 16

EQ_PANEL_WIDTH_RATIOS = (1.0, 1.5, 1.5)
EQ_PANEL_GAP_AB = 0.18
EQ_PANEL_GAP_BC = 0.18

print('MODEL_CFG =', cfg_to_dict(MODEL_CFG))

## 3. CIFAR repeated fits and random direct APJN points

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PLOT_TICK_FS = 16
PLOT_LABEL_FS = 18
PLOT_TITLE_FS = 18
PLOT_ANNOTATION_FS = 18
PLOT_ALPHA_LEGEND_FS = 16

In [ ]:
MODEL_CFG = ModelConfig(
    model_name='vit_base_patch16_224',
    depth=64,
    num_classes=100,
    img_size=224,
    replace_gelu_with_relu=True,
    inplace_relu=False,
    seed=0,
)
MEAN_FIELD_CFG = build_mean_field_cfg_for_vit_base(depth=MODEL_CFG.depth)

In [ ]:

RESULTS_POSTFIX = ""
SAVE_RESULTS_ROOT = "/content/drive/MyDrive/ml_projects/mapes_variance"
SAVE_FIT_HIST_RESULTS = True
SAVE_RANDOM_DIRECT_RESULTS = True
REWRITE = False

In [ ]:
# Plot.
# Load/prepare once, then re-plot from `fit_scatter_plot_data`.
ATTN_MULT = 2.0
MLP_MULT = 1.0

# fit_hist_bundle = f"/content/drive/MyDrive/ml_projects/mapes_variance/fit_hist_depth64_stride8_determ_seed_124_mask_False_num_inits_8/results.pkl"
# attn and mlp mults
fit_hist_bundle = f"/content/drive/MyDrive/ml_projects/mapes_variance/fit_hist_depth64_stride8_determ_seed_124_mask_False_num_inits_8_attn_mult_{ATTN_MULT}_mlp_mult_{MLP_MULT}/results.pkl"

random_inverse_bundle = "/content/drive/MyDrive/ml_projects/mapes_variance/random_direct_depth32_layers5_test/results.pkl"
random_direct_bundle = "/content/drive/MyDrive/ml_projects/mapes_variance/random_direct_depth32_layers5_test/results.pkl"
# random_inverse_bundle = "/content/drive/MyDrive/ml_projects/mapes_variance/random_inverse_depth64_layers8/results.pkl"
# random_direct_bundle = "/content/drive/MyDrive/ml_projects/mapes_variance/random_direct_depth64_layers8_rescaled_preln_weight_2/results.pkl"

fit_scatter_plot_data = prepare_fit_and_scatter_plot_data(
    fit_hist_bundle,
    random_inverse_bundle,
    random_direct_bundle,
)

In [ ]:
# for i in range(150):
#   print(fit_scatter_plot_data['fit_bundle']['samples'][i]['direct_fit']['preln_scale_C'])

In [ ]:
DEFAULT_ALPHAS = tuple(np.arange(0.1, 1.9 + 1e-9, 0.2).astype(float))

DEFAULT_Q0_GRID = tuple(np.linspace(0.0, 1.6, 101))
# DEFAULT_P0_GRID = tuple(np.union1d(np.linspace(-0.2, 0, 61), np.linspace(0, 1.6, 101)))
DEFAULT_RHO0_GRID = tuple(np.linspace(-0.4, 1.0, 151))
# DEFAULT_P0_GRID = tuple(np.linspace(0.0, 0.0, 1))
DEFAULT_PRELN_SCALE_GRID = tuple(np.geomspace(0.1, 20, 400))

MEAN_FIELD_CFG = build_mean_field_cfg_for_vit_base(
    depth=MODEL_CFG.depth,
    attn_mult=ATTN_MULT,
    mlp_mult=MLP_MULT,
    )

fit_scatter_plot_data_refit = refit_fit_scatter_plot_data(
    fit_scatter_plot_data,
    MEAN_FIELD_CFG,
    n_tokens=196,
    q0_values=DEFAULT_Q0_GRID,
    rho0_values=DEFAULT_RHO0_GRID,
    c_values=DEFAULT_PRELN_SCALE_GRID,
    fit_pre_ln=True,
    mask_all_p_values=False,
)

In [ ]:
print(fit_scatter_plot_data_refit['fit_bundle']['direct_mape'][27])

In [ ]:
mapes = fit_scatter_plot_data_refit['fit_bundle']['direct_mape']
print(sum(mapes)/len(mapes))

In [ ]:
vals = [fit_scatter_plot_data_refit['fit_bundle']['samples'][i]['direct_fit']['p0'] for i in range(90)]

In [ ]:
fig_diag = plot_inverse_fit_sample_diagnostics(
    fit_scatter_plot_data,
    sample_index=27,
    style_cfg=STYLE_CFG,
    apjn_direction='direct'
)

In [ ]:
restored_stats = collect_restored_cifar_block0_dot_stats(
    MODEL_CFG,
    batch_seed=124,
    sample_index=0,
    block_indices=[0, 16, 32, 48, 60],
    use_derf=True,
    alpha = 1.0,
    std_threshold=0.0,
    max_epochs_to_search=20,
)

In [ ]:
a = plot_restored_cifar_block0_dot_histograms(restored_stats)

In [ ]:
FIT_PANEL_COL_GAP = 0.18
FIT_PANEL_ROW_GAP = 0.16
FIT_LOWER_TO_CBAR_GAP = 0.1
FIT_SCATTER_POINT_ALPHA = 0.7
FIT_PERCENTILE_LABEL_ALPHA = 0.7
FIT_PERCENTILE_LABEL_FS = 12
PANEL_D_REDUCED_GAP = 0.2

STYLE_CFG = FinalThreePanelStyleConfig(
    title_fs=20,
    label_fs=20,
    annotation_fs=20,
    colorbar_pad=0.0,
    )

fig_hist_scatter = plot_fit_and_scatter_figure(
    fit_scatter_plot_data_refit,
    STYLE_CFG,
    panel_col_gap=FIT_PANEL_COL_GAP,
    panel_row_gap=FIT_PANEL_ROW_GAP,
    lower_row_to_colorbar_gap=FIT_LOWER_TO_CBAR_GAP,
    tick_fs=PLOT_TICK_FS,
    label_fs=PLOT_LABEL_FS,
    alpha_legend_fs=PLOT_ALPHA_LEGEND_FS,
    title_fs=PLOT_TITLE_FS,
    percentile_annotation_fs=FIT_PERCENTILE_LABEL_FS,
    percentile_annotation_alpha=FIT_PERCENTILE_LABEL_ALPHA,
    scatter_point_alpha=FIT_SCATTER_POINT_ALPHA,
    legend_width_scale=0.72,
    legend_height_scale=0.4
)

**MAPE 4-panel plots**

In [ ]:
MULT_PAIRS = [(1.0, 1.0), (2.0, 1.0), (1.0, 2.0), (2.0, 2.0)]
PANEL_LAYOUT = {
    (1.0, 2.0): (0, 0),
    (2.0, 2.0): (0, 1),
    (1.0, 1.0): (1, 0),
    (2.0, 1.0): (1, 1),
}

MAPE_PANEL_TITLE_FS = PLOT_TITLE_FS
MAPE_LABEL_FS = PLOT_LABEL_FS
MAPE_TICK_FS = PLOT_TICK_FS
MAPE_SECTION_TITLE_FS = PLOT_TITLE_FS + 2
MAPE_SECTION_TITLE_Y = 0.985
MAPE_SUBPLOTS_TOP = 0.92

def _panel_title(mult_pair):
    mlp_mult, attn_mult = mult_pair
    return rf"$(\sigma_{{21}}, \sigma_{{OV}})=({mlp_mult**2 * 0.6:.1f},\ {attn_mult**2 * 0.3:.1f})$"

def _fit_hist_bundle_path(mult_pair):
    mlp_mult, attn_mult = mult_pair
    if mult_pair == (1.0, 1.0):
        return "/content/drive/MyDrive/ml_projects/mapes_variance/fit_hist_depth64_stride8_determ_seed_124_mask_False_num_inits_8/results.pkl"
    return (
        "/content/drive/MyDrive/ml_projects/mapes_variance/"
        f"fit_hist_depth64_stride8_determ_seed_124_mask_False_num_inits_8_attn_mult_{attn_mult}_mlp_mult_{mlp_mult}/results.pkl"
    )

def _load_refit_plot_data(mult_pair):
    mlp_mult, attn_mult = mult_pair
    fit_scatter_plot_data = prepare_fit_and_scatter_plot_data(
        _fit_hist_bundle_path(mult_pair),
        random_inverse_bundle,
        random_direct_bundle,
    )
    mean_field_cfg = build_mean_field_cfg_for_vit_base(
        depth=MODEL_CFG.depth,
        attn_mult=attn_mult,
        mlp_mult=mlp_mult,
    )
    fit_scatter_plot_data_refit = refit_fit_scatter_plot_data(
        fit_scatter_plot_data,
        mean_field_cfg,
        n_tokens=196,
        q0_values=DEFAULT_Q0_GRID,
        rho0_values=DEFAULT_RHO0_GRID,
        c_values=DEFAULT_PRELN_SCALE_GRID,
        fit_pre_ln=True,
        mask_all_p_values=False,
    )
    return fit_scatter_plot_data_refit

def _plot_mape_panel(ax, mape_values, *, color, title):
    mape_pct = 100.0 * np.asarray(mape_values, dtype=float)
    x_vals = _swarm_x_positions(
        mape_pct.size,
        center=0.0,
        width=0.42,
        rng=np.random.default_rng(0),
    )
    ax.scatter(x_vals, mape_pct, color=color, s=28, alpha=0.8, edgecolors="none")
    for percentile, label in [(80, r"$80\%$ of data points"), (90, r"$90\%$ of data points")]:
        y_val = float(np.nanpercentile(mape_pct, percentile))
        ax.axhline(y_val, color=color, ls="--", lw=1.2, alpha=0.75)
        ax.text(
            0.98,
            y_val,
            label,
            ha="right",
            va="bottom",
            transform=ax.get_yaxis_transform(),
            fontsize=FIT_PERCENTILE_LABEL_FS,
            alpha=float(FIT_PERCENTILE_LABEL_ALPHA),
        )
    ax.set_title(title, fontsize=MAPE_PANEL_TITLE_FS)
    ax.set_xticks([])
    prettify_axes(ax)
    ax.tick_params(labelsize=MAPE_TICK_FS)
    return mape_pct

refit_plot_data_by_mult = {
    mult_pair: _load_refit_plot_data(mult_pair)
    for mult_pair in MULT_PAIRS
}

inverse_mape_pct_by_mult = {
    mult_pair: 100.0 * np.asarray(plot_data['fit_bundle']['inverse_mape'], dtype=float)
    for mult_pair, plot_data in refit_plot_data_by_mult.items()
}
direct_mape_pct_by_mult = {
    mult_pair: 100.0 * np.asarray(plot_data['fit_bundle']['direct_mape'], dtype=float)
    for mult_pair, plot_data in refit_plot_data_by_mult.items()
}

inverse_ylim_top = 1.05 * max(np.nanmax(vals) for vals in inverse_mape_pct_by_mult.values())
direct_ylim_top = 1.05 * max(np.nanmax(vals) for vals in direct_mape_pct_by_mult.values())

fig_mape_grid = plt.figure(figsize=(18, 10.5))
gs = fig_mape_grid.add_gridspec(
    2,
    5,
    width_ratios=[1.0, 1.0, 0.18, 1.0, 1.0],
    hspace=0.32,
    wspace=0.25,
)

inverse_axes = {
    (1.0, 2.0): fig_mape_grid.add_subplot(gs[0, 0]),
    (2.0, 2.0): fig_mape_grid.add_subplot(gs[0, 1]),
    (1.0, 1.0): fig_mape_grid.add_subplot(gs[1, 0]),
    (2.0, 1.0): fig_mape_grid.add_subplot(gs[1, 1]),
}
direct_axes = {
    (1.0, 2.0): fig_mape_grid.add_subplot(gs[0, 3]),
    (2.0, 2.0): fig_mape_grid.add_subplot(gs[0, 4]),
    (1.0, 1.0): fig_mape_grid.add_subplot(gs[1, 3]),
    (2.0, 1.0): fig_mape_grid.add_subplot(gs[1, 4]),
}

for mult_pair, ax in inverse_axes.items():
    _plot_mape_panel(
        ax,
        refit_plot_data_by_mult[mult_pair]['fit_bundle']['inverse_mape'],
        color="#4c78a8",
        title=_panel_title(mult_pair),
    )
    ax.set_ylim(0.0, inverse_ylim_top)

for mult_pair, ax in direct_axes.items():
    _plot_mape_panel(
        ax,
        refit_plot_data_by_mult[mult_pair]['fit_bundle']['direct_mape'],
        color="#2ca02c",
        title=_panel_title(mult_pair),
    )
    ax.set_ylim(0.0, direct_ylim_top)

for mult_pair, ax in inverse_axes.items():
    row_idx, col_idx = PANEL_LAYOUT[mult_pair]
    if col_idx == 0:
        ax.set_ylabel(r"MAPE, $\%$", fontsize=MAPE_LABEL_FS)
    if row_idx == 1:
        ax.set_xlabel("")

for mult_pair, ax in direct_axes.items():
    row_idx, col_idx = PANEL_LAYOUT[mult_pair]
    if col_idx == 0:
        ax.set_ylabel(r"MAPE, $\%$", fontsize=MAPE_LABEL_FS)
    if row_idx == 1:
        ax.set_xlabel("")

fig_mape_grid.text(
    0.24,
    MAPE_SECTION_TITLE_Y,
    r"(a) $\mathcal{J}^{\, B, b}$ fits (inverse)",
    ha="center",
    va="top",
    fontsize=MAPE_SECTION_TITLE_FS,
)
fig_mape_grid.text(
    0.76,
    MAPE_SECTION_TITLE_Y,
    r"(b) $\mathcal{J}^{\, b, 0}$ fits (direct)",
    ha="center",
    va="top",
    fontsize=MAPE_SECTION_TITLE_FS,
)
fig_mape_grid.subplots_adjust(top=MAPE_SUBPLOTS_TOP)
fig_mape_grid